<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/vision/vit-components/vit-components.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("vit-components", ...)`


# Vision Transformer Building Blocks: Patch Embedding, Position-Embedding Interpolation, DropPath

**Difficulty**: 🟡 Medium

### Problem Statement

`v3/modern-architectures/vit-mae` builds a ViT out of `nn.TransformerEncoderLayer` and pretrains it with MAE. This problem opens up the pieces that a production ViT recipe (DeiT, DINOv2, the vision tower of a VLM) adds around the transformer, the ones interviewers ask about:

- **Patch embedding.** ViT (Dosovitskiy et al., ICLR 2021) cuts the image into $P \times P$ patches, flattens each one and projects it linearly. That is exactly a `Conv2d` with kernel = stride = $P$. With $x_{rc}$ the patch at grid row $r$, column $c$:
  $$e_{rc} = W\,\mathrm{vec}(x_{rc}) + b, \qquad \text{token index } n = r \cdot \tfrac{W_{\text{img}}}{P} + c \quad\text{(row-major)}.$$
  You will write it both ways: as the conv, and as `unfold` + one matmul with the conv's own weight.
- **Position-embedding interpolation.** ViT's position embeddings are a learned table with one row per grid cell (plus one for the `[CLS]` token). Models are pretrained at 224 px and then fine-tuned or evaluated at 384 px with the same patch size, which means more patches and a table of the wrong size. The standard fix (ViT §3.2's "2D interpolation"; bicubic in DeiT, MAE and timm) treats the grid rows as a $D$-channel image, resamples it, and leaves the prefix tokens (`[CLS]`, register tokens) untouched.
- **Stochastic depth (DropPath).** During training, the output of a residual branch is dropped for randomly chosen **samples**, so for those samples the block is the identity and the network is effectively shallower. Kept samples are rescaled so the expectation is unchanged, and eval mode is the identity. With $m_b \sim \mathrm{Bernoulli}(1-p)$ drawn once per sample:
  $$\mathrm{DropPath}(h)_b = \frac{m_b}{1-p}\, h_b, \qquad \mathbb{E}\big[\mathrm{DropPath}(h)\big] = h.$$
  Huang et al. (ECCV 2016) proposed stochastic depth with one draw per mini-batch and a rescaling at test time. The per-sample form with the $1/(1-p)$ rescaling during training is timm's `drop_path` (torchvision's `stochastic_depth(mode="row")`), and it is what DeiT and DINO use. (DINOv2 uses it too, but for drop rates above 0.1 it switches to a variant that runs the branch on a random subset of the batch only.)
- **LayerScale** (CaiT, Touvron et al., ICCV 2021): a learnable per-channel gain $\gamma \in \mathbb{R}^D$ on each residual branch, initialised small, so that every block starts close to the identity. A pre-norm ViT block (ViT eqs. 2–3, plus LayerScale and DropPath) is
  $$x \leftarrow x + \mathrm{DropPath}\big(\gamma_1 \odot \mathrm{MSA}(\mathrm{LN}_1(x))\big), \qquad x \leftarrow x + \mathrm{DropPath}\big(\gamma_2 \odot \mathrm{MLP}(\mathrm{LN}_2(x))\big).$$

You will build these pieces and a small ViT classifier from them, then move a checkpoint to a higher, non-square resolution.

---

### Requirements

1. **`PatchEmbed(img_size=224, patch_size=16, in_chans=3, embed_dim=768)`**, an `nn.Module`:
   - `img_size` is an `int` (square) or an `(H, W)` tuple. Store `grid_size = (H // P, W // P)` as a tuple and `num_patches = (H // P) * (W // P)`.
   - The projection is `self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)`.
   - `forward(x)`: `(B, C, H, W) -> (B, N, D)` with tokens in **row-major** grid order. It must accept any `H`, `W` that are multiples of `P`, not only `img_size`.
2. **`patchify_linear(x, weight, bias, patch_size) -> Tensor`**: the same map without a convolution. `weight` is a Conv2d weight of shape `(D, C, P, P)`, and `bias` is `(D,)` or `None`. Extract the patches with `F.unfold` (or reshape/permute) and apply **one** matmul. The result must equal `conv2d(x, weight, bias, stride=P)` flattened to `(B, N, D)`, so the order of the values inside each patch vector must match how you reshape the weight.
3. **`interpolate_pos_embed(pos_embed, old_grid, new_grid, num_prefix_tokens=1) -> Tensor`**:
   - `pos_embed` is `(1, num_prefix_tokens + gh * gw, D)`: the prefix rows first, then the grid rows in row-major order. `old_grid = (gh, gw)` and `new_grid = (nh, nw)` are tuples and may be non-square.
   - Copy the prefix rows unchanged. Resample the grid as a `(1, D, gh, gw)` image with `F.interpolate(..., size=(nh, nw), mode="bicubic", align_corners=False)` (the DeiT/MAE recipe, which leaves `antialias` at its default, `False`), and return `(1, num_prefix_tokens + nh * nw, D)` with the new grid row-major.
   - Do not modify the input. The same grid in and out must give the input back.
   - Note: timm's `resample_abs_pos_embed` passes `antialias=True`. That switches PyTorch to a different (PIL-style) bicubic kernel, so its values differ from DeiT/MAE's even when upsampling. The tests and the grader accept either setting.
4. **`DropPath(drop_prob=0.0)`**, an `nn.Module` that stores `self.drop_prob`:
   - Identity (return `x` itself) in eval mode or when `drop_prob == 0`.
   - In training, draw one Bernoulli(`keep_prob = 1 - drop_prob`) value per sample from the global RNG, with mask shape `(B, 1, ..., 1)` for an input of any rank. Divide kept samples by `keep_prob`, except when `keep_prob == 0`: `drop_prob = 1` zeroes every sample and must not produce NaN.
5. **`ViTBlock(dim, num_heads, mlp_ratio=4.0, drop_path=0.0, init_values=None)`**, an `nn.Module` mapping `(B, N, dim) -> (B, N, dim)` (batch first):
   - Pre-norm, as in the formula above: `LayerNorm` → multi-head self-attention, then `LayerNorm` → MLP (`Linear(dim, int(dim * mlp_ratio))` → GELU → `Linear(..., dim)`), each added to the residual stream.
   - For attention, use `nn.MultiheadAttention(dim, num_heads, batch_first=True)` or a `qkv` linear with `F.scaled_dot_product_attention`. Attention itself is practised elsewhere.
   - **LayerScale**: if `init_values is not None` (including `0.0`), multiply each branch output by its own learnable `nn.Parameter` $\gamma$ of shape `(dim,)`, filled with `init_values`. With `None` there is no LayerScale. (timm's `Block` tests `if init_values:`, so `0.0` turns LayerScale *off* there. This problem uses `is not None`.)
   - **DropPath** with probability `drop_path` on each branch output, never on the residual stream.
6. **`ViT(img_size=224, patch_size=16, in_chans=3, num_classes=1000, embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0, drop_path_rate=0.0, init_values=None)`** with timm's attribute names:
   - `patch_embed` (your `PatchEmbed`); `cls_token`, an `nn.Parameter` of shape `(1, 1, D)`; `pos_embed`, an `nn.Parameter` of shape `(1, 1 + N, D)` that covers the CLS token as well; `blocks` (an `nn.ModuleList` or `nn.Sequential` of `ViTBlock`); `norm` (the final `LayerNorm`); `head` (`Linear(D, num_classes)`).
   - `forward(x)`: patch tokens → prepend CLS → add `pos_embed` → blocks → `norm` → `head` applied to the **CLS token** → `(B, num_classes)`.
   - Stochastic-depth decay rule: block `i` uses `drop_path = torch.linspace(0, drop_path_rate, depth)[i]`, so the rate rises linearly from 0 at the first block to `drop_path_rate` at the last.
   - A `ViT` is built for one resolution. To run a checkpoint at another one, build a new `ViT(img_size=...)` and load the old `state_dict` with `pos_embed` resampled by `interpolate_pos_embed`. Every other weight is resolution-independent.

---

### Constraints

- ✅ `nn.Conv2d` in `PatchEmbed`, `F.unfold` in `patchify_linear`, `F.interpolate` in `interpolate_pos_embed`, and `nn.MultiheadAttention` or `F.scaled_dot_product_attention` for attention.
- ❌ No convolution in `patchify_linear`.
- ❌ No `nn.TransformerEncoderLayer`. It hides the residual structure this problem is about, and it has no place for LayerScale or DropPath.
- ❌ No timm, and no `torchvision.ops.StochasticDepth`.
- ❌ No Python loops over patches or samples.

---

<details>
  <summary>💡 Hint</summary>

  - Print the shape of `self.proj(x)`. Which dimensions must be flattened, and which one must end up last?
  - `F.unfold(x, kernel_size=P, stride=P)` returns `(B, C*P*P, N)`. In what order are the $C \cdot P \cdot P$ values of one column laid out? Compare with `weight.reshape(D, -1)`.
  - The grid rows of `pos_embed` are stored as `(gh * gw, D)`. Is `reshape(1, D, gh, gw)` the same as reshaping to `(1, gh, gw, D)` and then permuting?
  - A DropPath mask that broadcasts over every dimension except the batch has how many elements?
  - With `init_values = 0.0` and both branches multiplied by zero, what must a pre-norm block return? What would a post-norm block return?

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
torch.manual_seed(0)
x_img = torch.randn(2, 3, 16, 24)      # 2 non-square "images", 16 x 24 pixels
P = 4                                   # patch size: a 4 x 6 grid of 24 patches


def patches_loop(x, weight, bias, p):
    # Reference: token r * (W/p) + c = <weight, patch at grid row r, column c> + bias.
    B, C, H, W = x.shape
    gh, gw = H // p, W // p
    out = torch.zeros(B, gh * gw, weight.shape[0])
    for r in range(gh):
        for c in range(gw):
            patch = x[:, :, r * p:(r + 1) * p, c * p:(c + 1) * p].reshape(B, -1)   # (B, C*p*p)
            out[:, r * gw + c] = patch @ weight.reshape(weight.shape[0], -1).T + bias
    return out


def bicubic_matrix(n_in, n_out, align_corners=False, a=-0.75, antialias=False):
    # Reference (n_out, n_in) matrix of 1-D bicubic resampling, as in F.interpolate(mode="bicubic"):
    # align_corners=False puts output sample i at input coordinate (i + 0.5) * n_in / n_out - 0.5
    # (pixel CENTRES line up); align_corners=True at i * (n_in - 1) / (n_out - 1) (corner pixels
    # line up). Keys' cubic kernel with a = -0.75; taps that fall outside the grid are clamped.
    # antialias=True (timm's setting; align_corners=False only) is PIL's filter instead: a = -0.5,
    # the kernel stretched by n_in / n_out when shrinking, out-of-grid taps dropped and the
    # remaining weights renormalised to sum to 1.
    def cubic(t, a=a):
        t = abs(t)
        if t <= 1:
            return ((a + 2) * t - (a + 3)) * t * t + 1
        return ((a * t - 5 * a) * t + 8 * a) * t - 4 * a if t < 2 else 0.0

    m = torch.zeros(n_out, n_in)
    if antialias:
        s = max(n_in / n_out, 1.0)
        for i in range(n_out):
            centre = (i + 0.5) * n_in / n_out
            lo, hi = max(int(centre - 2 * s + 0.5), 0), min(int(centre + 2 * s + 0.5), n_in)
            w = torch.tensor([cubic((j - centre + 0.5) / s, a=-0.5) for j in range(lo, hi)])
            m[i, lo:hi] = w / w.sum()
        return m
    for i in range(n_out):
        if align_corners:
            src = i * (n_in - 1) / (n_out - 1) if n_out > 1 else 0.0
        else:
            src = (i + 0.5) * n_in / n_out - 0.5
        f = math.floor(src)
        for k in range(-1, 3):
            m[i, min(max(f + k, 0), n_in - 1)] += cubic(k - (src - f))
    return m


def close_to_bicubic(rows, grid, new_grid):
    # rows: (nh * nw, D) resampled grid rows; grid: the (gh, gw, D) input grid. True if rows match
    # bicubic resampling with antialias=False (DeiT / MAE) or antialias=True (timm).
    (gh, gw, d), (nh, nw) = grid.shape, new_grid
    return any(torch.allclose(rows, torch.einsum("ih,hwd,jw->ijd", bicubic_matrix(gh, nh, antialias=aa), grid,
                                                 bicubic_matrix(gw, nw, antialias=aa)).reshape(-1, d), atol=1e-5)
               for aa in (False, True))

In [ ]:
class PatchEmbed(nn.Module):
    """Image -> sequence of patch tokens, via a Conv2d with kernel = stride = patch_size."""

    def __init__(self, img_size=224, patch_size: int = 16, in_chans: int = 3, embed_dim: int = 768):
        super().__init__()
        # TODO: grid_size (tuple), num_patches, self.proj = nn.Conv2d(...)
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, C, H, W) with H, W multiples of patch_size.

        Returns:
            (B, N, D) tokens, N = (H/P) * (W/P), token r * (W/P) + c = patch at row r, column c.
        """
        # TODO
        raise NotImplementedError


def patchify_linear(x: torch.Tensor, weight: torch.Tensor, bias, patch_size: int) -> torch.Tensor:
    """
    The patch embedding as unfold + one matmul (no convolution).

    Args:
        x: (B, C, H, W).
        weight: (D, C, P, P) Conv2d weight.
        bias: (D,) or None.
        patch_size: P.

    Returns:
        (B, N, D), equal to conv2d(x, weight, bias, stride=P).flatten(2).transpose(1, 2).
    """
    # TODO: patches as (B, N, C*P*P) vectors, weight as (D, C*P*P) in the SAME order, one matmul
    raise NotImplementedError

In [ ]:
pe = PatchEmbed(img_size=(16, 24), patch_size=P, in_chans=3, embed_dim=10)
assert pe.grid_size == (4, 6) and pe.num_patches == 24, (pe.grid_size, pe.num_patches)
tokens = pe(x_img)
assert tokens.shape == (2, 24, 10), tokens.shape
with torch.no_grad():
    ref = patches_loop(x_img, pe.proj.weight, pe.proj.bias, P)
assert torch.allclose(tokens, ref, atol=1e-5), "token r*(W/P) + c must be the patch at row r, column c"
assert pe(torch.randn(1, 3, 8, 20)).shape == (1, 10, 10), "any H, W that are multiples of P"

with torch.no_grad():
    lin = patchify_linear(x_img, pe.proj.weight, pe.proj.bias, P)
assert torch.allclose(lin, tokens, atol=1e-5), "unfold + matmul must equal the Conv2d"
w, b = torch.randn(5, 3, 3, 3), torch.randn(5)
x15 = x_img[:, :, :15]                                   # 15 x 24 with P = 3: a 5 x 8 grid
assert torch.allclose(patchify_linear(x15, w, b, 3),
                      F.conv2d(x15, w, b, stride=3).flatten(2).transpose(1, 2), atol=1e-5)
print("PatchEmbed / patchify_linear tests passed")

In [ ]:
def interpolate_pos_embed(pos_embed: torch.Tensor, old_grid, new_grid,
                          num_prefix_tokens: int = 1) -> torch.Tensor:
    """
    Resample learned position embeddings to a new patch grid (DeiT / MAE fine-tuning recipe).

    Args:
        pos_embed: (1, num_prefix_tokens + gh * gw, D); prefix rows first, then the grid row-major.
        old_grid: (gh, gw).
        new_grid: (nh, nw).
        num_prefix_tokens: leading rows that are not on the grid (CLS, registers); copied as-is.

    Returns:
        (1, num_prefix_tokens + nh * nw, D).
    """
    # TODO: split prefix / grid; grid -> (1, D, gh, gw); bicubic, align_corners=False,
    #       antialias=False; back to (1, nh*nw, D); concatenate the prefix in front
    raise NotImplementedError

In [ ]:
D = 8
pos = torch.randn(1, 1 + 4 * 6, D)                     # CLS + a 4 x 6 grid (row-major)
assert torch.allclose(interpolate_pos_embed(pos, (4, 6), (4, 6)), pos, atol=1e-6), "same grid -> unchanged"

new = interpolate_pos_embed(pos, (4, 6), (8, 9))
assert new.shape == (1, 1 + 8 * 9, D), new.shape
assert torch.equal(new[:, :1], pos[:, :1]), "the CLS row must be copied, not interpolated"
grid = pos[0, 1:].reshape(4, 6, D)                      # (h, w, D)
assert close_to_bicubic(new[0, 1:], grid, (8, 9)), "values differ from the bicubic reference"

# CLS + 4 register tokens; a non-square grid shrunk along h and stretched along w.
pos5 = torch.randn(1, 5 + 6 * 3, D)
out5 = interpolate_pos_embed(pos5, (6, 3), (2, 5), num_prefix_tokens=5)
assert out5.shape == (1, 5 + 2 * 5, D) and torch.equal(out5[:, :5], pos5[:, :5])
assert close_to_bicubic(out5[0, 5:], pos5[0, 5:].reshape(6, 3, D), (2, 5))
print("interpolate_pos_embed tests passed")

In [ ]:
class DropPath(nn.Module):
    """Stochastic depth per sample: drop a residual branch for random samples, rescale the rest."""

    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, ...) branch output.

        Returns:
            Same shape. Eval mode or drop_prob == 0: x itself. Training: each sample is zeroed with
            probability drop_prob, otherwise divided by keep_prob = 1 - drop_prob.
        """
        # TODO: one Bernoulli(keep_prob) draw per sample, mask shape (B, 1, ..., 1)
        raise NotImplementedError

In [ ]:
x = torch.rand(64, 5, 6) + 0.5
dp = DropPath(0.25)
dp.eval()
assert torch.equal(dp(x), x), "identity in eval mode"
dp.train()
torch.manual_seed(0)
out = dp(x)
dropped = (out == 0).flatten(1).all(1)
kept = torch.isclose(out, x / 0.75).flatten(1).all(1)
assert (dropped | kept).all(), "each sample must be dropped entirely, or kept entirely and divided by 0.75"
print(f"dropped {int(dropped.sum())} of 64 samples (expected about 16)")

torch.manual_seed(0)
big = DropPath(0.3).train()(torch.ones(20000, 3))
assert abs(big.mean().item() - 1.0) < 0.03, "E[DropPath(x)] must equal x: divide by keep_prob"
assert abs((big[:, 0] == 0).float().mean().item() - 0.3) < 0.02, "drop with probability drop_prob"
assert (DropPath(1.0).train()(x) == 0).all() and torch.equal(DropPath(0.0).train()(x), x)
print("DropPath tests passed")

In [ ]:
class LayerScale(nn.Module):
    """Learnable per-channel gain gamma (shape (dim,)) on a residual branch (CaiT)."""

    def __init__(self, dim: int, init_values: float = 1e-5):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError


class ViTBlock(nn.Module):
    """Pre-norm transformer block: x + DropPath(LS(Attn(LN(x)))), then x + DropPath(LS(MLP(LN(x))))."""

    def __init__(self, dim: int, num_heads: int, mlp_ratio: float = 4.0, drop_path: float = 0.0,
                 init_values=None):
        super().__init__()
        # TODO: norm1, attention, LayerScale (only if init_values is not None), DropPath,
        #       norm2, MLP (Linear -> GELU -> Linear), LayerScale, DropPath
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, N, dim) -> (B, N, dim)."""
        # TODO
        raise NotImplementedError

In [ ]:
torch.manual_seed(1)
tokens = torch.randn(2, 5, 16)
blk = ViTBlock(16, num_heads=4).eval()
assert blk(tokens).shape == tokens.shape

# Pre-norm: every branch sees LayerNorm(x), which ignores a constant added to all channels, and the
# residual stream carries that constant through untouched. A post-norm block would not.
blk64 = ViTBlock(16, 4).double().eval()
x64 = tokens.double()
with torch.no_grad():
    assert torch.allclose(blk64(x64 + 3.0), blk64(x64) + 3.0, atol=1e-8), "use x + f(LN(x)), not LN(x + f(x))"

# Zero-initialised LayerScale: the block starts as the identity (init_values = 0.0 still means "on").
zero = ViTBlock(16, 4, init_values=0.0)
assert torch.equal(zero(tokens), tokens)

# drop_path = 1.0: in training both branches are always dropped; in eval DropPath does nothing.
always = ViTBlock(16, 4, drop_path=1.0).train()
assert torch.equal(always(tokens), tokens), "DropPath belongs on the branch, not on x + branch"
never = ViTBlock(16, 4, drop_path=0.0)
never.load_state_dict(always.state_dict())
with torch.no_grad():
    assert torch.allclose(always.eval()(tokens), never.eval()(tokens), atol=1e-6)
print("ViTBlock tests passed")

In [ ]:
class ViT(nn.Module):
    """Patch embedding + CLS token + learned position embedding + pre-norm blocks + head on CLS."""

    def __init__(self, img_size=224, patch_size: int = 16, in_chans: int = 3, num_classes: int = 1000,
                 embed_dim: int = 768, depth: int = 12, num_heads: int = 12, mlp_ratio: float = 4.0,
                 drop_path_rate: float = 0.0, init_values=None):
        super().__init__()
        # TODO: patch_embed, cls_token (1, 1, D), pos_embed (1, 1 + N, D), blocks with
        #       drop_path = linspace(0, drop_path_rate, depth)[i], norm, head
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, C, H, W) at the resolution the model was built for -> (B, num_classes) logits."""
        # TODO
        raise NotImplementedError

In [ ]:
torch.manual_seed(2)
vit = ViT(img_size=32, patch_size=8, num_classes=7, embed_dim=32, depth=4, num_heads=4, drop_path_rate=0.3)
assert vit.pos_embed.shape == (1, 1 + 16, 32) and vit.cls_token.shape == (1, 1, 32)
imgs = torch.randn(3, 3, 32, 32)
assert vit.eval()(imgs).shape == (3, 7)
rates = [[round(m.drop_prob, 3) for m in blk.modules() if isinstance(m, DropPath)] for blk in vit.blocks]
print("DropPath rates per block:", rates)
assert all(abs(r - want) < 1e-6 for rs, want in zip(rates, torch.linspace(0, 0.3, 4).tolist()) for r in rs)
vit.train()
imgs16 = torch.randn(16, 3, 32, 32)
assert not torch.allclose(vit(imgs16), vit(imgs16)), "drop_path_rate > 0: two training passes differ"

# Fine-tune at a higher, non-square resolution: build the new model, resample pos_embed, load.
state = {k: v.clone() for k, v in vit.state_dict().items()}
state["pos_embed"] = interpolate_pos_embed(state["pos_embed"], (4, 4), (6, 8))
vit_hr = ViT(img_size=(48, 64), patch_size=8, num_classes=7, embed_dim=32, depth=4, num_heads=4)
vit_hr.load_state_dict(state)          # strict: every other weight is resolution-independent
with torch.no_grad():
    logits = vit_hr.eval()(torch.randn(2, 3, 48, 64))
assert logits.shape == (2, 7) and torch.isfinite(logits).all()
print(f"loaded the 32x32 checkpoint into a 48x64 model: {vit_hr.patch_embed.num_patches} patches, logits {tuple(logits.shape)}")
print("ViT tests passed")

In [ ]:
# Look: why align_corners=False. Keep the patch size and enlarge the image: new patch j has its
# centre at old-grid coordinate (j + 0.5) * old / new - 0.5, the half-pixel mapping of
# align_corners=False. Resample an embedding that is a smooth function of the patch centre and
# compare with that function evaluated at the new patch centres.
freqs = torch.linspace(0.2, 0.9, 8)


def smooth_pos(gh, gw, sh=1.0, sw=1.0):
    r = (torch.arange(gh) + 0.5) * sh                     # patch centres in old-grid units
    c = (torch.arange(gw) + 0.5) * sw
    R, C = torch.meshgrid(r, c, indexing="ij")
    return (torch.sin(freqs * R[..., None]) + torch.cos(1.3 * freqs * C[..., None])).reshape(1, gh * gw, 8)


old, new = (6, 6), (12, 9)
pos = torch.cat([torch.zeros(1, 1, 8), smooth_pos(*old)], dim=1)
truth = smooth_pos(*new, sh=old[0] / new[0], sw=old[1] / new[1])[0]
ours = interpolate_pos_embed(pos, old, new)[0, 1:]
grid = pos[0, 1:].reshape(*old, 8)
corners = torch.einsum("ih,hwd,jw->ijd", bicubic_matrix(6, 12, align_corners=True), grid,
                       bicubic_matrix(6, 9, align_corners=True)).reshape(-1, 8)
print("mean |error| against the embedding at the new patch centres:")
print(f"  align_corners=False (patch centres line up): {(ours - truth).abs().mean():.4f}")
print(f"  align_corners=True  (grid corners line up):  {(corners - truth).abs().mean():.4f}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("vit-components", PatchEmbed=PatchEmbed, patchify_linear=patchify_linear,
      interpolate_pos_embed=interpolate_pos_embed, DropPath=DropPath, ViTBlock=ViTBlock, ViT=ViT)